# Import Modules

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tqdm import tqdm
import os
import matplotlib.pyplot as plt
import seaborn as sns
import math
from torch.utils.tensorboard import SummaryWriter 
import json
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import squareform
from sklearn.manifold import spectral_embedding
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import rbf_kernel

## Import Dataset Class

In [3]:
from dataset_classes import AT

## Import Models

In [4]:
from models_with_temporal_graph import GLFN_TC_Attention, GLFN_TC_GlobalLocal, GLFN_TC_Linear, GLFN_TC_MultiScale

## Import Training and Testing Loops

In [5]:
from helper_functions_trial import train_model, test_model

## Train and Test

### MultiScale

## GCN

GCN_Layer: 1, Hidden_Dimension: 64, Kernel_Size: 7, Dilation: 3

In [ ]:
def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"

    dataset = AT(
        csv_path=r"GLFN-TC\Datasets\AT-Dataset\AT Dataset.csv",
        T_in=72,
        T_out=240,
        lag_hours=[1,12,24,168], 
        rolling_windows=[12,24],
    )

    # --- 1. Define splits based on RAW data length ---
    total_len = len(dataset.df_numeric)
    train_split_idx = int(0.6 * total_len)
    val_split_idx = int(0.8 * total_len)
    
    print(f"Raw data length: {total_len}")
    print(f"Scaler 'train_size' (raw rows): {train_split_idx}")
    print(f"Scaler 'val_end' (raw rows): {val_split_idx}")

    # --- 2. Fit scaler only on the raw TRAIN subset ---
    scaler = StandardScaler()
    scaler.fit(dataset.df_numeric.iloc[:train_split_idx].values.astype(np.float32))

    # --- 3. Apply same scaler to all data ---
    dataset.apply_scaler(scaler)
    dataset.scaler = scaler  # keep for later inverse-transform

    train_end = train_split_idx - dataset.T_in
    val_end = val_split_idx - dataset.T_in
    
    # Get the total number of *valid* samples
    effective_len = len(dataset)
    
    # Ensure our calculated indices don't exceed the total number of samples
    train_end = min(train_end, effective_len)
    val_end = min(val_end, effective_len)

    train_idx = range(0, train_end)
    val_idx = range(train_end, val_end)
    test_idx = range(val_end, effective_len)

    print(f"Total valid samples: {effective_len}")
    print(f"Train samples: {len(train_idx)}, Val samples: {len(val_idx)}, Test samples: {len(test_idx)}")
    
    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)
    test_subset = Subset(dataset, test_idx)

    # Base hyperparameters configuration
    hparams = {
        "N": dataset.N,
        "T_in": 72,
        "T_out": 240,
        "d": 32,
        "hidden_dim": 64, 
        "GCN_Layer": 5,
        "dropout_forecast": 0.1,
        "dropout_gcn": 0.2,
        "dropout_temporal": 0.2,
        "lr": 1e-4,
        "scheduler_patience": 3,
        "batch_size": 64, 
        "epochs": 100,
        "weight_decay": 1e-4, 
    }
    
    # --- DataLoaders (Kept outside loop since the data structure does not change) ---
    train_loader = DataLoader(train_subset, batch_size=hparams["batch_size"], shuffle=False)
    val_loader = DataLoader(val_subset, batch_size=hparams["batch_size"], shuffle=False)
    test_loader = DataLoader(test_subset, batch_size=hparams["batch_size"], shuffle=False)
    print(f"\n🚚 DataLoaders ready. Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")
    
    # --- Define Baselines ---
    baseline_ks = 7
    baseline_dil = 3

    # --- Define Independent Experiment Configurations ---
    # This loop runs once for kernel size changes and once for dilation changes smoothly.
    experiments = [
        # 1. Loop over Kernel Size changes (with dilation fixed at baseline 3)
        {"kernel_size": 3, "dilation": baseline_dil, "type": "KS_Sweep"},
        {"kernel_size": 5, "dilation": baseline_dil, "type": "KS_Sweep"},
        {"kernel_size": 11, "dilation": baseline_dil, "type": "KS_Sweep"},
        
        # 2. Loop over Dilation changes (with kernel size fixed at baseline 7)
        {"kernel_size": baseline_ks, "dilation": 1, "type": "Dil_Sweep"},
        {"kernel_size": baseline_ks, "dilation": 2, "type": "Dil_Sweep"},
        {"kernel_size": baseline_ks, "dilation": 5, "type": "Dil_Sweep"},
    ]

    # --- Sequential Experiment Execution ---
    for exp in experiments:
        ks = exp["kernel_size"]
        dil = exp["dilation"]
        exp_type = exp["type"]
        
        # Dynamically update current loop targets inside hparams dict for logging integrity
        hparams["kernel_size"] = ks
        hparams["dilation"] = dil
        
        # Unique naming convention per target configuration execution
        run = f"TR-GNN-Multi_Scale_AT_{exp_type}_GCN_{hparams['GCN_Layer']}_HD_{hparams['hidden_dim']}_KS_{ks}_Dil_{dil}"
        
        print("\n" + "="*70)
        print(f"🎬 STARTING EXPERIMENT ({exp_type}): KS={ks} | Dil={dil}")
        print(f"Run Directory Name: {run}")
        print("="*70 + "\n")
        
        log_dir = f"TR_GNN_AT/{run}"  
        writer = SummaryWriter(log_dir)
        
        # Log active setup configurations parameters to TensorBoard
        writer.add_text("hparams", json.dumps(hparams, indent=2))
        
        # --- Initialize Fresh Model Instance ---
        model = GLFN_TC_MultiScale(
            N=hparams["N"],
            T_in=hparams["T_in"],
            T_out=hparams["T_out"],
            d=hparams["d"],
            hidden_dim=hparams["hidden_dim"],
            GCN_Layer=hparams["GCN_Layer"],
            dropout_gcn=hparams["dropout_gcn"],
            dropout_temporal=hparams["dropout_temporal"],
            kernel_size=ks,
            dilation=dil,
        ).to(device)
        
        model = train_model(
            model,
            train_loader,
            val_loader,
            epochs=hparams["epochs"],
            lr=hparams["lr"],
            device=device,
            scheduler_patience=hparams["scheduler_patience"],
            writer=writer,
            weight_decay=hparams["weight_decay"],
            save_path=f"Final_Run/{run}_best_model.pth"
        )

        print(f"\n🧪 Testing performance for configuration run: {run}...\n")
        preds, trues = test_model(
            dataset=dataset,
            model=model, 
            test_loader=test_loader,
            device=device,
            writer=writer
        )
        
        writer.close()
        print(f"🏁 Finished execution for run: {run}\n")
        
if __name__ == "__main__":
    main()